# MultiHandler

Multi-task synchronized acquisition with hardware triggers.

In [ ]:
# Check nidaqmx availability
try:
    import nidaqmx
except ImportError:
    raise RuntimeError(
        "nidaqmx is not installed. Install with: pip install nidaqmx"
    )

In [ ]:
import numpy as np
from nidaqwrapper import MultiHandler, AITask, AOTask, list_devices

# List connected hardware
devices = list_devices()
print(f"Found {len(devices)} device(s):")
for i, dev in enumerate(devices):
    print(f"  Device {i}: {dev['name']} ({dev['product_type']})")

## Basic Multi-Task

Synchronize multiple AITask objects with hardware triggers.

In [ ]:
# Edit device/terminal names to match your hardware (cDAQ module + trigger line)
ai_device = "cDAQ1Mod1"               # NI-DAQmx module name string
clock = "/cDAQ1Mod1/ai/SampleClock"   # shared sample-clock terminal
trigger = "/cDAQ1/PFI0"               # shared start-trigger terminal
n_samples = 5000                      # finite burst length per channel

# Master analog task: finite acquisition on the shared sample clock
task1 = AITask(task_name='accel_task', sample_rate=10000)
task1.add_channel(channel_name='accel_x', device=ai_device, channel_ind=0,
                  max_val=10.0, min_val=-10.0, units='V')
task1.configure(sample_mode='finite', samples_per_channel=n_samples,
                clock_source=clock)
task1.set_start_trigger(trigger, edge='rising')

# Slave analog task: same clock, same trigger
task2 = AITask(task_name='force_task', sample_rate=10000)
task2.add_channel(channel_name='force_z', device=ai_device, channel_ind=1,
                  max_val=100.0, min_val=-100.0, units='V')
task2.configure(sample_mode='finite', samples_per_channel=n_samples,
                clock_source=clock)
task2.set_start_trigger(trigger, edge='rising')

# Pass the resolved nidaqmx tasks (.task) so MultiHandler keeps the finite +
# trigger configuration instead of re-applying default continuous timing.
multi = MultiHandler()
success = multi.configure(input_tasks=[task1.task, task2.task])
print(f"Configuration successful: {success}")

if success:
    connected = multi.connect()
    print(f"Connected: {connected}")

    if connected:
        # Hardware-trigger path: arms all tasks, then waits for the trigger
        print("Waiting for hardware trigger...")
        data = multi.acquire()

        # Data structure: {task_name: {channel_name: ndarray}}
        print(f"\nAcquired data from {len(data)} tasks:")
        for task_name, channels in data.items():
            print(f"  {task_name}:")
            for ch_name, ch_data in channels.items():
                print(f"    {ch_name}: {ch_data.shape}")

        multi.disconnect()

## Input + Output

Synchronized input and output tasks.

In [ ]:
# Edit device/terminal names to match your hardware
ai_device = "cDAQ1Mod1"
ao_device = "cDAQ1Mod2"
clock = "/cDAQ1Mod1/ai/SampleClock"   # shared sample-clock terminal
trigger = "/cDAQ1/PFI0"               # shared start-trigger terminal
n_samples = 10000

# Input task: finite acquisition sharing the sample clock + start trigger
ai = AITask(task_name='sync_input', sample_rate=10000)
ai.add_channel('sensor', device=ai_device, channel_ind=0,
               max_val=10.0, min_val=-10.0, units='V')
ai.configure(sample_mode='finite', samples_per_channel=n_samples,
             clock_source=clock)
ai.set_start_trigger(trigger, edge='rising')

# Output task: finite generation (buffer sized at construction), same trigger
ao = AOTask(task_name='sync_output', sample_rate=10000,
            samples_per_channel=n_samples)
ao.add_channel('stimulus', device=ao_device, channel_ind=0,
               max_val=10.0, min_val=-10.0)
ao.configure(clock_source=clock)
ao.set_start_trigger(trigger, edge='rising')

# Configure with both input and output (pass resolved .task objects)
multi = MultiHandler()
multi.configure(input_tasks=[ai.task], output_tasks=[ao.task])
multi.connect()

print(f"Configured {len(multi.input_tasks)} input and {len(multi.output_tasks)} output tasks")
print(f"Trigger type: {multi.trigger_type}")

multi.disconnect()

## Validation

MultiHandler validates task compatibility during configure().

In [ ]:
print("MultiHandler validation checks:")
print("\n1. Type validation - all tasks must be AITask, AOTask, nidaqmx.Task, or str")
print("2. Task resolution - str task names loaded from NI MAX")
print("3. Validity check - tasks must be open with at least one channel")
print("4. Sample rate consistency - all tasks in a group must share the same rate")
print("5. Timing validation - clock source and timing config must match")
print("6. Trigger consistency - all tasks must use the same trigger type/source")
print("7. Acquisition mode - FINITE for hardware triggers, CONTINUOUS for software")
print("\nIf any validation fails, configure() returns False.")

## Cleanup with disconnect()

`MultiHandler` has no context manager — always release hardware explicitly
with `disconnect()` (idempotent; safe to call from a `finally` block).

In [ ]:
# Manual cleanup with disconnect()
task = AITask(task_name='cleanup_demo', sample_rate=10000)
task.add_channel('ch0', device="cDAQ1Mod1", channel_ind=0,
                 max_val=10.0, min_val=-10.0, units='V')

multi = MultiHandler()
multi.configure(input_tasks=[task])
multi.connect()

# ... do work ...

multi.disconnect()
print("Disconnected and cleaned up.")

In [ ]:
# Alternative: use disconnect() in a try/finally
task = AITask(task_name='cleanup_demo', sample_rate=10000)
task.add_channel('ch0', device="cDAQ1Mod1", channel_ind=0,
                 max_val=10.0, min_val=-10.0, units='V')

multi = MultiHandler()
try:
    multi.configure(input_tasks=[task])
    multi.connect()
    # ... do work ...
finally:
    multi.disconnect()
    print("Cleanup complete.")